# Gate 2 — A1 (MBHT + B1) SMOKE TEST trên RetailRocket

## ⚠ ĐỌC TRƯỚC: notebook này KHÔNG quyết định B1 sống hay chết

RetailRocket ở đây là **smoke test hạ tầng**, không phải phép thử khoa học.

**Tiêu chí PASS (chỉ 3 điều này):**
1. Không NaN / Inf trong training.
2. Số param tăng **đúng 1,984** (= |T| 31 × hidden 64).
3. Chạy lại A0 với `enable_transition_embedding=0` reproduce đúng số cũ.

**Metric A1-vs-A0 trên RetailRocket KHÔNG phải tiêu chí go/no-go — chỉ log lại, không kết luận.**

Lý do (đã kiểm chứng trên dữ liệu thật, không phải phỏng đoán): test set của RetailRocket
**không chứa một sự kiện mua nào**. Train có 20 loại transition, test chỉ có 6, và
`CART→BUY` — transition mang ý nghĩa nặng nhất cho luận điểm của B1 — chiếm 6.68% trong
train nhưng **0% trong test**. Tín hiệu mà B1 nhắm tới gần như không tồn tại lúc inference
trên bộ này. Tmall/IJCAI thì có đủ (6.85% / 7.82% sự kiện mua ở test).

**Go/no-go thật sự đánh trên Tmall**, ở notebook kế tiếp.

| | train | test |
|---|---:|---:|
| số transition phân biệt | 20 | 6 |

Commit ghim: `b005ba6`

## 1. GPU + mount Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## 2. Đường dẫn (giống Gate 1 — dùng lại data/checkpoint đã có trên Drive)

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/DeAnThS'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'dataset')
CKPT_DIR     = os.path.join(PROJECT_ROOT, 'checkpoints')
LOG_DIR      = os.path.join(PROJECT_ROOT, 'logs')
RUN_META_DIR = os.path.join(PROJECT_ROOT, 'run_meta')
CODE_DIR     = '/content/MBHT-KDD22'

for d in [DATA_DIR, CKPT_DIR, LOG_DIR, RUN_META_DIR]:
    os.makedirs(d, exist_ok=True)

DATASET_ROOT = os.path.join(DATA_DIR, 'MBHT_dataset')
assert os.path.isdir(DATASET_ROOT), f'Chua co {DATASET_ROOT} -- chay notebook Gate 1-2 truoc.'

REPO_URL      = 'https://github.com/nguyenlmhcm/MBHT-KDD22.git'
BRANCH        = 'bt-mbht'
PINNED_COMMIT = 'b005ba69ab14e0e5bfb31e323f3a4083d231e65e'   # B1 landed
DATASET_NAME  = 'retail_beh'

print('DATASET_ROOT:', DATASET_ROOT)
print('PINNED_COMMIT:', PINNED_COMMIT)

## 3. Clone code đã có B1 + cài dependency

In [ ]:
import subprocess, time

if os.path.isdir(CODE_DIR):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)

for attempt in range(1, 6):
    r = subprocess.run(['git','clone','-b',BRANCH,REPO_URL,CODE_DIR], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'Clone OK (attempt {attempt})'); break
    print(f'attempt {attempt} failed:', r.stderr.strip())
    if os.path.isdir(CODE_DIR):
        subprocess.run(['rm','-rf',CODE_DIR], check=True)
    if attempt == 5:
        raise RuntimeError('git clone failed 5x')
    time.sleep(10*attempt)

subprocess.run(['git','-C',CODE_DIR,'checkout',PINNED_COMMIT], check=True)
head = subprocess.run(['git','-C',CODE_DIR,'rev-parse','HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head == PINNED_COMMIT, f'{head} != {PINNED_COMMIT}'
print('commit:', head)

In [ ]:
%cd {CODE_DIR}
!pip install -q hyperopt pandas tqdm scikit_learn pyyaml colorlog colorama tensorboard

## 4. Chạy unit test B1 trên chính máy Colab

29 test này đã pass trên VPS; chạy lại ở đây để chắc chắn không có gì vỡ do khác phiên bản thư viện.

In [ ]:
!python tests/test_transition_utils.py
print()
!python tests/test_b1_integration.py

## 5. Load dataset + in mapping behavior (xác nhận n_types = 5, |T| = 31)

In [ ]:
from recbole.config import Config
from recbole.data import create_dataset
from recbole.model.transition_utils import transition_vocab_size

base_cfg = {
    'data_path': DATASET_ROOT,
    'USER_ID_FIELD': 'session_id',
    'load_col': None,
    'neg_sampling': None,
    'benchmark_filename': ['train', 'test'],
    'alias_of_item_id': ['item_id_list'],
    'MAX_ITEM_LIST_LENGTH': 200,
}
schema_config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=base_cfg)
dataset = create_dataset(schema_config)

type_map = dataset.field2token_id['item_type_list']
n_types  = len(type_map)
print('field2token_id[item_type_list] =', type_map)
print('n_types =', n_types)
print('|T|     =', transition_vocab_size(n_types))
assert n_types == 5, f'RetailRocket phai co 5 type, dang thay {n_types}'
assert transition_vocab_size(n_types) == 31

## 6. Bảng tần suất transition (checklist 15.1)

Chạy trên chính train dataloader, qua đúng hàm model dùng.

In [ ]:
import collections, torch
from recbole.data.utils import get_dataloader, create_samplers
from recbole.model.transition_utils import (
    build_transition_seq, count_transitions, format_transition_table,
)

id2token = dataset.field2id_token['item_type_list']
counter = collections.Counter()

train_ds, test_ds = dataset.build()
tmp_cfg = Config(model='MBHT', dataset=DATASET_NAME, config_dict={**base_cfg, 'train_batch_size': 64, 'eval_batch_size': 128})
tr_sampler, te_sampler = create_samplers(tmp_cfg, dataset, [train_ds, test_ds])
tmp_loader = get_dataloader(tmp_cfg, 'train')(tmp_cfg, train_ds, tr_sampler, shuffle=False)

for batch in tmp_loader:
    item_seq = batch['item_id_list']
    type_seq = batch['item_type_list']
    counter = count_transitions(
        build_transition_seq(type_seq, item_seq > 0, n_types), counter=counter
    )

print(format_transition_table(counter, n_types, id2token))

## 7. Hàm chạy một run (dùng chung cho A0-rerun và A1)

Mọi thứ giống hệt Gate 1 trừ đúng một biến: `enable_transition_embedding`.

In [ ]:
import glob, json
from logging import getLogger
from recbole.model.sequential_recommender.mbht import MBHT
from recbole.utils import init_logger, init_seed, get_trainer, set_color

FROZEN = {
    'USER_ID_FIELD': 'session_id',
    'load_col': None,
    'neg_sampling': None,
    'benchmark_filename': ['train', 'test'],
    'alias_of_item_id': ['item_id_list'],
    'topk': [5, 10, 101],
    'metrics': ['Recall', 'NDCG', 'MRR'],
    'valid_metric': 'NDCG@10',
    'eval_args': {'mode': 'full', 'order': 'TO'},
    'MAX_ITEM_LIST_LENGTH': 200,
    'train_batch_size': 64,
    'eval_batch_size': 128,
    'hyper_len': 6,
    'scales': [5, 4, 20],
    'enable_hg': 1,
    'enable_ms': 1,
    'customized_eval': 1,
    'abaltion': '',
}

def run(tag, enable_b1, seed=2020):
    ckpt = os.path.join(CKPT_DIR, tag); os.makedirs(ckpt, exist_ok=True)
    logd = os.path.join(LOG_DIR, tag);  os.makedirs(logd, exist_ok=True)

    cfg_dict = {**FROZEN,
                'data_path': DATASET_ROOT,
                'checkpoint_dir': ckpt,
                'gpu_id': 0,
                'seed': seed,
                'enable_transition_embedding': int(enable_b1)}
    config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=cfg_dict)
    init_seed(config['seed'], config['reproducibility'])
    init_logger(config, log_root=logd)
    logger = getLogger()
    logger.info(f'TAG={tag} commit={PINNED_COMMIT} enable_transition_embedding={int(enable_b1)}')

    tr, te = dataset.build()
    s_tr, s_te = create_samplers(config, dataset, [tr, te])
    train_data = get_dataloader(config, 'train')(config, tr, s_tr, shuffle=True)
    test_data  = get_dataloader(config, 'test')(config, te, s_te, shuffle=False)

    model = MBHT(config, train_data.dataset).to(config['device'])
    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{tag}] enable_transition_embedding={model.enable_transition_embedding} '
          f'n_behavior_types={model.n_behavior_types} |T|={model.n_transitions} '
          f'params={n_params}')

    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)
    prev = sorted(glob.glob(os.path.join(ckpt, '*.pth')), key=os.path.getmtime)
    if prev:
        print(f'[{tag}] resuming from {prev[-1]}')
        trainer.resume_checkpoint(prev[-1])

    score, result = trainer.fit(train_data, test_data, saved=True, show_progress=config['show_progress'])
    print(set_color(f'[{tag}] test result', 'yellow') + f': {result}')

    with open(os.path.join(RUN_META_DIR, f'{tag}.json'), 'w') as f:
        json.dump({'tag': tag, 'commit': PINNED_COMMIT, 'seed': seed,
                   'enable_transition_embedding': int(enable_b1),
                   'n_params': n_params, 'result': {k: float(v) for k, v in result.items()}},
                  f, indent=2)
    return n_params, result

## 8. PASS #2 — param delta đúng 1,984

Dựng cả hai model (chưa train) và so số param.

In [ ]:
cfg_off = Config(model='MBHT', dataset=DATASET_NAME,
                 config_dict={**FROZEN, 'data_path': DATASET_ROOT, 'seed': 2020,
                              'enable_transition_embedding': 0})
cfg_on  = Config(model='MBHT', dataset=DATASET_NAME,
                 config_dict={**FROZEN, 'data_path': DATASET_ROOT, 'seed': 2020,
                              'enable_transition_embedding': 1})

tr, te = dataset.build()
s_tr, s_te = create_samplers(cfg_off, dataset, [tr, te])
probe = get_dataloader(cfg_off, 'train')(cfg_off, tr, s_tr, shuffle=False)

init_seed(2020, True); m_off = MBHT(cfg_off, probe.dataset)
init_seed(2020, True); m_on  = MBHT(cfg_on,  probe.dataset)

p_off = sum(p.numel() for p in m_off.parameters())
p_on  = sum(p.numel() for p in m_on.parameters())
delta = p_on - p_off
print(f'A0 (flag=0) params : {p_off}')
print(f'A1 (flag=1) params : {p_on}')
print(f'delta              : {delta}   (ky vong 31*64 = {31*64})')
assert delta == 31*64, f'PASS #2 FAIL: delta={delta}'

new_keys = set(m_on.state_dict()) - set(m_off.state_dict())
print('key moi:', new_keys)
assert new_keys == {'transition_embedding.weight'}

# moi tham so A0 phai y nguyen khi bat flag (tru gating_bias -- xem canh bao cuoi notebook)
sd_off, sd_on = m_off.state_dict(), m_on.state_dict()
diff = [k for k in sd_off if k != 'gating_bias' and not torch.equal(sd_off[k], sd_on[k])]
print('tham so A0 bi lech khi bat flag:', diff)
assert not diff, f'bat B1 lam lech {diff} -- ablation bi nhieu'
print('\nPASS #2 OK')

## 9. PASS #3 — A0 rerun (flag=0) phải ra lại số cũ

Số Gate 1 để đối chiếu:
```
recall@5 0.9305  recall@10 0.9375  ndcg@5 0.9202  ndcg@10 0.9224  mrr@5 0.9167  mrr@10 0.9177
```
Chạy tag riêng để **không đè** checkpoint A0 cũ.

In [ ]:
GATE1_A0 = {'recall@5':0.9305,'recall@10':0.9375,'ndcg@5':0.9202,
            'ndcg@10':0.9224,'mrr@5':0.9167,'mrr@10':0.9177}

p_a0, res_a0 = run('A0rerun-MBHT-retail_beh-seed2020', enable_b1=False, seed=2020)

print('\n--- A0 rerun vs Gate 1 ---')
print(f'{"metric":>12} {"Gate1":>9} {"rerun":>9} {"delta":>10}')
for k, v in GATE1_A0.items():
    now = float(res_a0[k]); print(f'{k:>12} {v:>9.4f} {now:>9.4f} {now-v:>+10.4f}')

## 10. A1 — bật B1 (chỉ log, KHÔNG kết luận go/no-go)

In [ ]:
p_a1, res_a1 = run('A1-MBHT-B1-retail_beh-seed2020', enable_b1=True, seed=2020)

print('\n--- A1 vs A0 rerun (CHI DE THAM KHAO, khong phai go/no-go) ---')
print(f'{"metric":>12} {"A0":>9} {"A1":>9} {"delta":>10}')
for k in GATE1_A0:
    a0v, a1v = float(res_a0[k]), float(res_a1[k])
    print(f'{k:>12} {a0v:>9.4f} {a1v:>9.4f} {a1v-a0v:>+10.4f}')
print(f'\nparams: A0={p_a0}  A1={p_a1}  delta={p_a1-p_a0}')
print('\nNHAC LAI: RetailRocket test set khong co su kien mua nao (CART->BUY = 0%).')
print('Delta o tren KHONG dung de ket luan B1 tot hay khong. Go/no-go danh tren Tmall.')

## 11. Kết luận smoke test + cần dán về những gì

**Dán lại cho Claude:**
1. Output cell 4 (29 unit test — phải `20/20` và `9/9`).
2. Output cell 5 (`field2token_id`, n_types, |T|).
3. Bảng tần suất transition ở cell 6.
4. Output cell 8 (param delta — PASS #2).
5. Bảng A0 rerun vs Gate 1 ở cell 9 (PASS #3).
6. Bảng A1 vs A0 ở cell 10 (chỉ để log).
7. Bất kỳ NaN/traceback nào (PASS #1).

---

### ⚠ Một cảnh báo về khả năng tái lập, phát hiện lúc viết B1

`mbht.py` cấp phát `gating_bias` bằng `torch.Tensor(...)` — **bộ nhớ chưa khởi tạo** — và dòng
`nn.init.normal_(self.gating_bias, std=0.02)` bị **comment mất** (dòng 100), trong khi mọi tham
số anh em đều được khởi tạo. Đây là lỗi của code gốc, **không phải do B1**.

Hệ quả: `gating_bias` không chịu ảnh hưởng của seed. Đo trên VPS: cùng seed, absmax dao động từ
`2.3e-10` đến `0.12` giữa các lần dựng, làm output forward lệch tới `3.2e-05`; `torch.Tensor()`
trần từng trả về giá trị lớn tới `1.7e+28`.

**Nghĩa là A0 vốn đã không tái lập bit-exact**, kể cả trước khi có B1. Nếu cell 9 cho ra
delta nhỏ khác 0, nhiều khả năng là do đây chứ không phải do code B1.

Chưa sửa gì — chờ anh quyết định.